# Creating Data Analysis Reports with R

## What you'll learn
- How to use Jupyter notebooks as polished data analysis reports
- Summary statistics and formatted tables
- Cross-tabulations and pivot tables
- How to structure a report: introduction, data, analysis, findings
- How to export your notebook as HTML or PDF

## Prerequisites
- Completed Notebooks 01 (R Fundamentals) and 06 (Visualization)

## What Is a Data Analysis Report?

A data analysis report is a document that combines:
- **Narrative** — explanations of what you're doing and why
- **Code** — the actual analysis
- **Output** — tables, charts, and statistics
- **Findings** — what the data tells you

Jupyter notebooks are perfect for this because they naturally mix all four. The notebook **is** the report.

In this notebook, we'll build a mini-report analyzing the sales dataset, and learn the techniques along the way.

## Markdown Formatting Refresher

Markdown cells in Jupyter support rich formatting. Here's what you'll use most in reports:

```
# Heading 1
## Heading 2
### Heading 3

**bold text**
*italic text*

- Bullet point
- Another bullet

1. Numbered list
2. Another item

| Column A | Column B |
|----------|----------|
| value 1  | value 2  |

> Blockquote for callouts or key findings
```

Use these to structure your report sections and highlight key findings.

## Setup

In [ ]:
library(tidyverse)
library(knitr)

sales <- read_csv("../data/sales.csv", show_col_types = FALSE)
employees <- read_csv("../data/employees.csv", show_col_types = FALSE)

## Summary Statistics

The first thing to include in any report is an overview of the data: how much data do you have, and what do the key variables look like?

### Using `summary()`

In [ ]:
# Quick overview of the dataset
summary(sales)

`summary()` is useful for a quick look, but the output isn't very pretty. For a report, you'll want formatted tables.

### Custom Summary Tables with dplyr

Build your own summary tables with `summarize()`. This gives you full control over what to include.

In [ ]:
# Overall dataset summary
sales |>
  summarize(
    total_transactions = n(),
    date_range_start = min(date),
    date_range_end = max(date),
    total_revenue = sum(quantity * unit_price),
    avg_transaction_value = mean(quantity * unit_price),
    unique_products = n_distinct(product),
    unique_regions = n_distinct(region)
  )

In [ ]:
# Summary by category
category_summary <- sales |>
  mutate(revenue = quantity * unit_price) |>
  group_by(category) |>
  summarize(
    transactions = n(),
    total_revenue = round(sum(revenue), 2),
    avg_price = round(mean(unit_price), 2),
    avg_quantity = round(mean(quantity), 1)
  ) |>
  arrange(desc(total_revenue))

category_summary

### Formatted Tables with `knitr::kable()`

The `kable()` function from the `knitr` package renders data frames as clean, formatted tables. This looks much better in a report than raw R output.

In [ ]:
kable(
  category_summary,
  col.names = c("Category", "Transactions", "Total Revenue ($)", "Avg Price ($)", "Avg Quantity"),
  caption = "Sales Summary by Product Category",
  format.args = list(big.mark = ",")
)

## Using `count()` for Frequency Tables

`count()` is a shortcut for `group_by() |> summarize(n = n())`. It's great for quick frequency tables.

In [ ]:
# How many transactions per region?
sales |> count(region, sort = TRUE)

In [ ]:
# Top 10 most sold products
sales |>
  count(product, sort = TRUE) |>
  head(10) |>
  kable(col.names = c("Product", "Times Sold"), caption = "Top 10 Products by Transaction Count")

## Cross-Tabulations

A **cross-tabulation** (or cross-tab) shows the relationship between two categorical variables. For example: how many transactions occurred in each region for each category?

### Using `table()`

In [ ]:
# Cross-tab: category x region
table(sales$category, sales$region)

### Using tidyr::pivot_wider() for Clean Cross-Tabs

`pivot_wider()` converts long data to a wide format — perfect for cross-tabulation tables.

In [ ]:
# Revenue cross-tab: category (rows) x region (columns)
revenue_crosstab <- sales |>
  mutate(revenue = quantity * unit_price) |>
  group_by(category, region) |>
  summarize(total_revenue = round(sum(revenue), 0), .groups = "drop") |>
  pivot_wider(names_from = region, values_from = total_revenue, values_fill = 0)

kable(revenue_crosstab, caption = "Total Revenue ($) by Category and Region", format.args = list(big.mark = ","))

---

# Mini-Report: Sales Data Analysis

Below is an example of how a complete data analysis report might look. Notice how markdown cells provide context and interpretation between code cells.

## 1. Introduction

This report analyzes 500 retail transactions spanning two years (2023-2024) across five product categories and five geographic regions. The goal is to identify which categories and regions drive the most revenue, and to spot trends over time.

## 2. Data Overview

In [ ]:
cat("Dataset: 500 retail transactions\n")
cat("Date range:", min(sales$date), "to", max(sales$date), "\n")
cat("Categories:", paste(unique(sales$category), collapse = ", "), "\n")
cat("Regions:", paste(unique(sales$region), collapse = ", "), "\n")

In [ ]:
# Key metrics
overall <- sales |>
  summarize(
    `Total Transactions` = n(),
    `Total Revenue` = paste0("$", format(round(sum(quantity * unit_price), 2), big.mark = ",")),
    `Average Transaction` = paste0("$", round(mean(quantity * unit_price), 2)),
    `Unique Products` = n_distinct(product)
  )

kable(t(overall), col.names = c("Value"), caption = "Key Metrics")

## 3. Analysis

### 3.1 Revenue by Category

In [ ]:
cat_rev <- sales |>
  mutate(revenue = quantity * unit_price) |>
  group_by(category) |>
  summarize(
    transactions = n(),
    total_revenue = round(sum(revenue), 2),
    pct_of_total = round(sum(revenue) / sum(sales$quantity * sales$unit_price) * 100, 1)
  ) |>
  arrange(desc(total_revenue))

kable(cat_rev, col.names = c("Category", "Transactions", "Revenue ($)", "% of Total"))

In [ ]:
ggplot(cat_rev, aes(x = reorder(category, total_revenue), y = total_revenue, fill = category)) +
  geom_col() +
  coord_flip() +
  labs(title = "Total Revenue by Category", x = "", y = "Revenue ($)") +
  theme_minimal() +
  theme(legend.position = "none")

### 3.2 Regional Performance

In [ ]:
region_rev <- sales |>
  mutate(revenue = quantity * unit_price) |>
  group_by(region) |>
  summarize(
    transactions = n(),
    total_revenue = round(sum(revenue), 2),
    avg_transaction = round(mean(revenue), 2)
  ) |>
  arrange(desc(total_revenue))

kable(region_rev, col.names = c("Region", "Transactions", "Revenue ($)", "Avg Transaction ($)"))

In [ ]:
ggplot(region_rev, aes(x = reorder(region, total_revenue), y = total_revenue, fill = region)) +
  geom_col() +
  coord_flip() +
  labs(title = "Revenue by Region", x = "", y = "Revenue ($)") +
  theme_minimal() +
  theme(legend.position = "none")

### 3.3 Monthly Trends

In [ ]:
monthly <- sales |>
  mutate(
    month = as.Date(paste0(substr(date, 1, 7), "-01")),
    revenue = quantity * unit_price
  ) |>
  group_by(month) |>
  summarize(total_revenue = sum(revenue), transactions = n())

ggplot(monthly, aes(x = month, y = total_revenue)) +
  geom_line(color = "steelblue", linewidth = 1) +
  geom_point(color = "steelblue") +
  labs(title = "Monthly Revenue Trend", x = "Month", y = "Revenue ($)") +
  theme_minimal()

### 3.4 Category-Region Breakdown

In [ ]:
kable(revenue_crosstab, caption = "Revenue ($) by Category and Region", format.args = list(big.mark = ","))

## 4. Findings

Based on the analysis above, here are the key findings:

> **Finding 1:** The specific category and region breakdowns are determined by the synthetic data, but the structure of the analysis demonstrates how to present findings.

> **Finding 2:** Cross-tabulations reveal how performance varies across two dimensions simultaneously.

> **Finding 3:** Monthly trend analysis helps identify seasonal patterns or growth/decline.

## Exporting Your Report

Once your notebook is ready, you can export it as HTML or PDF from the terminal:

```bash
# Export to HTML
jupyter nbconvert --to html 07-reports.ipynb

# Export to PDF (requires LaTeX installed)
jupyter nbconvert --to pdf 07-reports.ipynb
```

In VS Code, you can also use the built-in export: click the `...` menu at the top of the notebook and select "Export As".

**Tip:** Before exporting, run all cells from top to bottom (`Run All`) to make sure everything executes cleanly. A report with error messages doesn't look professional.

## Report Structure Checklist

When creating your own reports, aim for this structure:

1. **Title and date** — what is this report about?
2. **Introduction** — what question are you answering? Why does it matter?
3. **Data overview** — what data are you using? How much? What time period?
4. **Analysis** — organized into subsections, each with:
   - A question or angle
   - The relevant code and output (tables, charts)
   - A brief interpretation
5. **Findings / Conclusion** — the main takeaways
6. **Appendix** (optional) — additional detail, methodology notes, or supplementary charts

---
## Summary

- Jupyter notebooks are natural report documents: narrative + code + output
- Use `kable()` to create clean formatted tables
- Use `count()`, `summarize()`, and `pivot_wider()` for summary and cross-tab tables
- Combine markdown sections with code cells and ggplot2 charts for a polished report
- Export with `jupyter nbconvert` or VS Code's built-in export

**Congratulations!** You've completed the R track. You now have the tools to load data, transform it, visualize it, work with big datasets, and present your findings in a professional report.